# Phase 3 — Foundation-Guided Self-Training on VME

The Phase 1 baseline gets 38.2 mAP50 on VME without ever seeing a VME label. This notebook tries to close the gap to the 55.6 oracle by self-training: the source detector pseudo-labels the unlabeled VME training images, and a student is fine-tuned on those labels with strong augmentation. The twist is that Grounding DINO (Phase 2) is fused into the pseudo-labels — it adds boxes the source detector missed, since its blind spots are different.

The method is the offline, single-stage version of Adaptive Teacher (Li et al., CVPR 2022): pseudo-labels are generated once on clean images, the student trains on distorted copies. Simpler than the full online EMA variant but stays entirely in the MMDetection/TOOD stack, which keeps the comparison to Phase 1 clean.

Two students are trained under identical conditions, differing only in pseudo-label source (fused vs. source-only), to isolate the effect of foundation guidance. Outputs go to `QCRI-CV/phase3/`.

---
# Section 1 — Environment
Run at the start of every session. Identical MMDetection stack to Phase 1 (the student is a TOOD detector). One runtime restart between 1.2 and 1.3; an A100 is required for training.

### 1.1 — GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

NVIDIA A100-SXM4-40GB, 40960 MiB


### 1.2 — PyTorch 2.4.0

In [ ]:
!pip uninstall -y -q torch torchvision torchaudio
!pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 142.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 112.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 61.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 135.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 21.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 47.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 10.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 13.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

**Restart the runtime now** (Runtime → Restart session), then continue from 1.3.

### 1.3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 1.4 — MMDetection stack

#### 1.4a — Fresh install

In [ ]:
!pip install -q mmengine
#!pip install -q mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html
!pip install -q mmdet==3.3.0 sahi wandb pycocotools rasterio fire

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.7/452.7 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 13.3 MB/s eta 0:00:00


#### 1.4b — Restore from cache

In [ ]:
import shutil, os, site

PKG_CACHE = '/content/drive/MyDrive/QCRI-CV/saved_packages'
SITE = site.getsitepackages()[0]
assert os.path.isdir(PKG_CACHE), 'No package cache on Drive — run 1.4a, then 1.8'
for item in os.listdir(PKG_CACHE):
    src, dst = os.path.join(PKG_CACHE, item), os.path.join(SITE, item)
    if os.path.exists(dst):
        shutil.rmtree(dst) if os.path.isdir(dst) else os.remove(dst)
    shutil.copytree(src, dst) if os.path.isdir(src) else shutil.copy2(src, dst)
print(f'Restored {len(os.listdir(PKG_CACHE))} packages')
!pip install -q mmengine mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html
!pip install -q mmdet==3.3.0 sahi wandb pycocotools rasterio fire

In [ ]:
# import just mmcv the one that takes 51 min to install
import shutil, os, site

PKG_CACHE = '/content/drive/MyDrive/QCRI-CV/saved_packages'
SITE = site.getsitepackages()[0]

for item in ['mmcv', 'mmcv-2.1.0.dist-info']:
    src, dst = os.path.join(PKG_CACHE, item), os.path.join(SITE, item)
    if not os.path.exists(src):
        print(f'Not in cache: {item}'); continue
    if os.path.exists(dst):
        shutil.rmtree(dst) if os.path.isdir(dst) else os.remove(dst)
    shutil.copytree(src, dst) if os.path.isdir(src) else shutil.copy2(src, dst)
    print(f'Restored: {item}')

Restored: mmcv
Restored: mmcv-2.1.0.dist-info


### 1.5 — MMDetection repository, pinned to v3.3.0

In [ ]:
import os
if not os.path.exists('/content/mmdetection'):
    !git clone -q --depth 1 -b v3.3.0 https://github.com/open-mmlab/mmdetection.git /content/mmdetection
print('mmdetection v3.3.0 ready')

Note: switching to '44ebd17b145c2372c4b700bfb9cb20dbd28ab64a'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

mmdetection v3.3.0 ready


### 1.6 — Pretrained backbone weights

In [ ]:
import os
BACKBONE = '/content/resnext101_64x4d-ee2c6f71.pth'
if not os.path.exists(BACKBONE):
    !wget -q --show-progress -O /content/resnext101_64x4d-ee2c6f71.pth https://huggingface.co/GeorgePearse/visdet-weights/resolve/main/openmmlab/resnext101_64x4d-ee2c6f71.pth
assert os.path.getsize(BACKBONE) // 2**20 > 100, 'Incomplete download — delete and retry'
print('Backbone weights ready')

/content/resnext101 100%[===================>] 311.38M  36.2MB/s    in 43s     
Backbone weights ready


### 1.7 — mmengine W&B patch

In [ ]:
import mmengine.visualization.vis_backend as vb
SRC = 'self._wandb.run.log_code(name=self._log_code_name)'
text = open(vb.__file__).read()
if SRC in text:
    open(vb.__file__, 'w').write(text.replace(SRC, 'pass  # log_code disabled'))
    print('Patched:', vb.__file__)
elif 'log_code disabled' in text:
    print('Already patched')
else:
    raise RuntimeError('log_code pattern not found — inspect vis_backend.py')

Patched: /usr/local/lib/python3.12/dist-packages/mmengine/visualization/vis_backend.py


/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


### 1.8 — Package cache (after a fresh install only)

In [ ]:
import shutil, os, site
PKG_CACHE = '/content/drive/MyDrive/QCRI-CV/saved_packages'
os.makedirs(PKG_CACHE, exist_ok=True)
SITE = site.getsitepackages()[0]
KEEP = ['mmcv', 'mmdet', 'mmengine', 'sahi', 'wandb', 'rasterio', 'pycocotools',
        'addict', 'yapf', 'terminaltables', 'shapely', 'click', 'fire']
n = 0
for item in os.listdir(SITE):
    if any(item.lower().startswith(k) for k in KEEP):
        src, dst = os.path.join(SITE, item), os.path.join(PKG_CACHE, item)
        try:
            if os.path.exists(dst):
                shutil.rmtree(dst) if os.path.isdir(dst) else os.remove(dst)
            shutil.copytree(src, dst) if os.path.isdir(src) else shutil.copy2(src, dst)
            n += 1
        except Exception as e:
            print('skipped', item, '->', e)
print(f'Cached {n} packages')

### 1.9 — Weights & Biases login

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: haytham-bd (haytham-bd-valeo) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### 1.10 — Environment verification

In [ ]:
import torch, mmdet, mmcv, mmengine, os
print(f'torch {torch.__version__} | {torch.cuda.get_device_name(0)} | mmdet {mmdet.__version__} | mmcv {mmcv.__version__}')
assert torch.__version__.startswith('2.4.0'), 'torch must be 2.4.0 — restart after 1.2'
assert mmcv.__version__ == '2.1.0'
assert os.path.exists('/content/resnext101_64x4d-ee2c6f71.pth'), 'Run 1.6'
import mmengine.visualization.vis_backend as vb
assert 'log_code disabled' in open(vb.__file__).read(), 'Run 1.7'
print('Environment ready')

torch 2.4.0+cu121 | NVIDIA A100-SXM4-40GB | mmdet 3.3.0 | mmcv 2.1.0
Environment ready


# Section 2 — Configuration
Paths, hyperparameters, and source/target balance. `SOURCE_FRACTION` controls how much of the xView tile set is kept as an anchoring signal; `REPEAT_VME` oversamples the pseudo-labeled target to balance the two streams. LR 0.0025 follows the Adaptive Teacher target-domain setting.

In [ ]:
import os, json, datetime

BASE = '/content/drive/MyDrive/QCRI-CV'
P1   = f'{BASE}/phase1'
P2   = f'{BASE}/phase2'
ROOT = f'{BASE}/phase3'

# inputs from earlier phases
P1_BASELINE_CFG  = f'{P1}/configs/tood_xview2vme_baseline.py'
P1_BASELINE_CKPT = f'{P1}/checkpoints/baseline/epoch_24.pth'
P1_RESULTS_JSON  = f'{P1}/results/phase1_results.json'
P2_GDINO_PSEUDO  = f'{P2}/pseudo_labels/vme_train_gdino_pseudo.json'
P2_RESULTS_JSON  = f'{P2}/results/phase2_results.json'

# labeled source (xView tiles from Phase 1) — anchors the student
XV_TILES_TRAIN_ANN = f'{P1}/xview_tiled/train_images/xview_tiled_train_coco.json'
XV_TILES_VAL_ANN   = f'{P1}/xview_tiled/val_images/xview_tiled_val_coco.json'
XV_TILES_TRAIN_DIR = f'{P1}/xview_tiled/train_images/'
XV_TILES_VAL_DIR   = f'{P1}/xview_tiled/val_images/'

# target (VME)
VME_TRAIN_ANN = f'{P1}/annotations/vme/train.json'
VME_VAL_ANN   = f'{P1}/annotations/vme/val.json'
VME_TEST_ANN  = f'{P1}/annotations/vme/test.json'
VME_IMGS      = f'{BASE}/VME_CDSI_datasets/satellite_images'

# outputs
PSEUDO   = f'{ROOT}/pseudo_labels'
SRC_SUB  = f'{ROOT}/source_subset'
CONFIGS  = f'{ROOT}/configs'
CKPTS    = f'{ROOT}/checkpoints'
RESULTS  = f'{ROOT}/results'
PREDS    = f'{RESULTS}/preds'
FIGS     = f'{RESULTS}/figures'
RESULTS_JSON = f'{RESULTS}/phase3_results.json'

TOOD_PSEUDO_ANN  = f'{PSEUDO}/vme_train_tood_pseudo.json'      # source detector only (ablation)
FUSED_PSEUDO_ANN = f'{PSEUDO}/vme_train_fused_pseudo.json'     # foundation-guided (main)
XV_TRAIN_SUBSET  = f'{SRC_SUB}/xview_tiled_train_subset.json'
XV_VAL_SUBSET    = f'{SRC_SUB}/xview_tiled_val_subset.json'

FG_CFG_PATH = f'{CONFIGS}/tood_selftrain_fused.py'
ST_CFG_PATH = f'{CONFIGS}/tood_selftrain_toodonly.py'
BACKBONE = '/content/resnext101_64x4d-ee2c6f71.pth'

for d in [PSEUDO, SRC_SUB, CONFIGS, CKPTS, RESULTS, PREDS, FIGS]:
    os.makedirs(d, exist_ok=True)

# pseudo-label generation
MIN_BOX          = 2       # drop pseudo boxes smaller than this (px) on either side
MAX_DETS_PER_IMG = 300     # cap pseudo boxes per image, kept by score
FUSE_IOU         = 0.5     # a GDINO box is "new" if it overlaps no source-detector box above this IoU

# student training (Adaptive-Teacher target-domain regime)
SOURCE_FRACTION  = 0.20    # fraction of xView tiles used to anchor the student
REPEAT_VME       = 3       # oversample the pseudo-labeled target to balance against source
SELFTRAIN_LR     = 0.0025
SELFTRAIN_EPOCHS = 8

def save_result(key, payload):
    db = json.load(open(RESULTS_JSON)) if os.path.exists(RESULTS_JSON) else {}
    payload = dict(payload); payload['_saved'] = datetime.datetime.now().isoformat(timespec='seconds')
    db[key] = payload
    with open(RESULTS_JSON, 'w') as f:
        json.dump(db, f, indent=2)
    print(f'saved -> {key}: {payload}')

def load_results():    return json.load(open(RESULTS_JSON)) if os.path.exists(RESULTS_JSON) else {}
def load_phase1():     return json.load(open(P1_RESULTS_JSON)) if os.path.exists(P1_RESULTS_JSON) else {}
def load_phase2():     return json.load(open(P2_RESULTS_JSON)) if os.path.exists(P2_RESULTS_JSON) else {}

for p in [P1_BASELINE_CFG, P1_BASELINE_CKPT, VME_TRAIN_ANN, VME_VAL_ANN, VME_TEST_ANN,
          XV_TILES_TRAIN_ANN, XV_TILES_VAL_ANN]:
    assert os.path.exists(p), f'Missing required input: {p}'
print('Output root:', ROOT)
print('GDINO pseudo-labels present:', os.path.exists(P2_GDINO_PSEUDO))
b = load_phase1().get('baseline_vme_test_epoch24', {}).get('mAP50', 'n/a')
o = load_phase1().get('oracle_vme_test_epoch24', {}).get('mAP50', 'n/a')
g = load_phase2().get('gdino_vme_test_best_prompt', {}).get('mAP50', 'n/a')
print(f'References -> baseline {b} | oracle {o} | gdino {g}')

Output root: /content/drive/MyDrive/QCRI-CV/phase3
GDINO pseudo-labels present: True
References -> baseline 38.23 | oracle 55.58 | gdino 14.68


# Section 3 — Pseudo-label generation
Threshold selection on VME val (3.1), inference on VME train (3.2), GDINO fusion (3.3), writing both pseudo-label files and checking density against the true annotation count (3.4), source subset (3.5).

### 3.1 — Confidence threshold selection
The detector runs on the VME validation set. Threshold is swept and selected by max F1 against the validation ground truth — trades precision (fewer noisy labels) against recall (more coverage).

In [ ]:
import numpy as np
from tqdm import tqdm
from mmdet.apis import init_detector, inference_detector
from pycocotools.coco import COCO

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

teacher = init_detector(P1_BASELINE_CFG, P1_BASELINE_CKPT, device='cuda:0')

coco_val = COCO(VME_VAL_ANN)
val_pred, val_gt = {}, {}
for info in tqdm(coco_val.loadImgs(coco_val.getImgIds()), desc='Teacher on VME val'):
    r = inference_detector(teacher, os.path.join(VME_IMGS, info['file_name']))
    val_pred[info['id']] = (r.pred_instances.bboxes.cpu().numpy(), r.pred_instances.scores.cpu().numpy())
    g = coco_val.loadAnns(coco_val.getAnnIds(imgIds=info['id']))
    val_gt[info['id']] = np.array([[x, y, x + w, y + h] for x, y, w, h in (a['bbox'] for a in g)]) if g else np.zeros((0, 4))

def pr_at(thr, match_iou=0.5):
    tp = fp = n_gt = 0
    for iid, (b, s) in val_pred.items():
        m = s >= thr
        keep, ks = b[m], s[m]
        gt = val_gt[iid]; n_gt += len(gt)
        if len(keep) == 0:
            continue
        if len(gt) == 0:
            fp += len(keep); continue
        iou = iou_matrix(keep, gt); matched = set()
        for i in np.argsort(-ks):
            j = int(np.argmax(iou[i]))
            if iou[i, j] >= match_iou and j not in matched:
                tp += 1; matched.add(j)
            else:
                fp += 1
    prec = tp / (tp + fp + 1e-9); rec = tp / (n_gt + 1e-9)
    return prec, rec, 2 * prec * rec / (prec + rec + 1e-9)

print(f"{'thr':>5} {'precision':>10} {'recall':>8} {'F1':>7}")
sweep = {}
for thr in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    sweep[thr] = pr_at(thr)
    print(f'{thr:>5.2f} {sweep[thr][0]:>10.3f} {sweep[thr][1]:>8.3f} {sweep[thr][2]:>7.3f}')

PSEUDO_THRESHOLD = max(sweep, key=lambda t: sweep[t][2])
print(f'\nSelected pseudo-label threshold (max F1 on val): {PSEUDO_THRESHOLD}')
save_result('pseudo_threshold_selection',
            {'threshold': PSEUDO_THRESHOLD, 'val_precision': round(sweep[PSEUDO_THRESHOLD][0], 3),
             'val_recall': round(sweep[PSEUDO_THRESHOLD][1], 3), 'val_f1': round(sweep[PSEUDO_THRESHOLD][2], 3)})

Loads checkpoint by local backend from path: /content/drive/MyDrive/QCRI-CV/phase1/checkpoints/baseline/epoch_24.pth


/usr/local/lib/python3.12/dist-packages/mmengine/runner/checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filename, map_location=map_l

06/17 09:59:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.0.conv2 is upgraded to version 2.
06/17 09:59:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.1.conv2 is upgraded to version 2.
06/17 09:59:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.2.conv2 is upgraded to version 2.
06/17 09:59:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.3.conv2 is upgraded to version 2.
06/17 09:59:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.0.conv2 is upgraded to version 2.
06/17 09:59:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.1.conv2 is upgraded to version 2.
06/17 09:59:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.2.conv2 is upgraded to version 2.
06/17 09:59:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.3.conv2 is upgraded to version 2.
06/17 09:59:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.4.conv2 is upgraded to version 2.
06/17 09:59:10 - mm

Teacher on VME val: 100%|██████████| 510/510 [08:04<00:00,  1.05it/s]

  thr  precision   recall      F1
 0.30      0.758    0.276   0.405
 0.40      0.858    0.163   0.274
 0.50      0.929    0.071   0.131
 0.60      0.961    0.023   0.044
 0.70      0.981    0.004   0.009
 0.80      1.000    0.000   0.001

Selected pseudo-label threshold (max F1 on val): 0.3
saved -> pseudo_threshold_selection: {'threshold': 0.3, 'val_precision': 0.758, 'val_recall': 0.276, 'val_f1': 0.405, '_saved': '2026-06-17T10:07:17'}


In [ ]:
print(f"{'thr':>5} {'precision':>10} {'recall':>8} {'F1':>7}")
sweep = {}
for thr in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    sweep[thr] = pr_at(thr)
    print(f'{thr:>5.2f} {sweep[thr][0]:>10.3f} {sweep[thr][1]:>8.3f} {sweep[thr][2]:>7.3f}')

PSEUDO_THRESHOLD = max(sweep, key=lambda t: sweep[t][2])
print(f'\nSelected pseudo-label threshold (max F1 on val): {PSEUDO_THRESHOLD}')
save_result('pseudo_threshold_selection',
            {'threshold': PSEUDO_THRESHOLD, 'val_precision': round(sweep[PSEUDO_THRESHOLD][0], 3),
             'val_recall': round(sweep[PSEUDO_THRESHOLD][1], 3), 'val_f1': round(sweep[PSEUDO_THRESHOLD][2], 3)})

  thr  precision   recall      F1
 0.05      0.283    0.549   0.374
 0.10      0.444    0.490   0.465
 0.15      0.546    0.434   0.483
 0.20      0.631    0.384   0.477
 0.25      0.697    0.334   0.451
 0.30      0.758    0.276   0.405
 0.40      0.858    0.163   0.274
 0.50      0.929    0.071   0.131
 0.60      0.961    0.023   0.044
 0.70      0.981    0.004   0.009
 0.80      1.000    0.000   0.001

Selected pseudo-label threshold (max F1 on val): 0.15
saved -> pseudo_threshold_selection: {'threshold': 0.15, 'val_precision': 0.546, 'val_recall': 0.434, 'val_f1': 0.483, '_saved': '2026-06-17T10:32:26'}


### 3.2 — Source-detector pseudo-labels on VME train
Detections above the selected threshold are kept, capped per image by score, and clipped to image bounds.

In [ ]:
coco_train = COCO(VME_TRAIN_ANN)
CAT_ID = int(coco_train.getCatIds()[0])

tood_boxes = {}
for info in tqdm(coco_train.loadImgs(coco_train.getImgIds()), desc='Teacher on VME train'):
    r = inference_detector(teacher, os.path.join(VME_IMGS, info['file_name']))
    b = r.pred_instances.bboxes.cpu().numpy()
    s = r.pred_instances.scores.cpu().numpy()
    m = s >= PSEUDO_THRESHOLD
    b, s = b[m], s[m]
    order = s.argsort()[::-1][:MAX_DETS_PER_IMG]
    W, H = info.get('width', 512), info.get('height', 512)
    boxes = []
    for (x1, y1, x2, y2), sc in zip(b[order], s[order]):
        x1, y1 = max(0.0, float(x1)), max(0.0, float(y1))
        x2, y2 = min(float(W), float(x2)), min(float(H), float(y2))
        if x2 - x1 >= MIN_BOX and y2 - y1 >= MIN_BOX:
            boxes.append(([x1, y1, x2, y2], float(sc)))
    tood_boxes[info['id']] = boxes
n_tood = sum(len(v) for v in tood_boxes.values())
print(f'Source-detector pseudo-boxes: {n_tood} over {len(tood_boxes)} images ({n_tood/len(tood_boxes):.1f} per image)')

loading annotations into memory...
Done (t=0.18s)
creating index...
index created!


Teacher on VME train: 100%|██████████| 2449/2449 [16:15<00:00,  2.51it/s]

Source-detector pseudo-boxes: 46668 over 2449 images (19.1 per image)


### 3.3 — Fuse the foundation detections
Grounding DINO boxes from Phase 2 that overlap no source-detector box (IoU below the fusion threshold) are added as complementary, higher-recall detections; the source detector's boxes are kept as-is.

In [ ]:
import json
gdino_by_img = {}
if os.path.exists(P2_GDINO_PSEUDO):
    for a in json.load(open(P2_GDINO_PSEUDO))['annotations']:
        x, y, w, h = a['bbox']
        gdino_by_img.setdefault(a['image_id'], []).append(([x, y, x + w, y + h], a.get('score', 1.0)))
    print(f"Loaded {sum(len(v) for v in gdino_by_img.values())} GDINO boxes")
else:
    print('GDINO pseudo-labels not found — fused set equals the source-only set')

fused_boxes, added = {}, 0
for iid, tb in tood_boxes.items():
    boxes = list(tb)
    t_arr = np.array([bx for bx, _ in tb]) if tb else np.zeros((0, 4))
    for gb, gs in gdino_by_img.get(iid, []):
        if len(t_arr) == 0 or iou_matrix(np.array([gb]), t_arr).max() < FUSE_IOU:
            boxes.append((gb, gs)); added += 1
    fused_boxes[iid] = boxes
n_fused = sum(len(v) for v in fused_boxes.values())
print(f'Fused pseudo-boxes: {n_fused} over {len(fused_boxes)} images ({n_fused/len(fused_boxes):.1f} per image); {added} added by the foundation model')

Loaded 2454 GDINO boxes
Fused pseudo-boxes: 47520 over 2449 images (19.4 per image); 852 added by the foundation model


### 3.4 — Write pseudo-label files and compare density to the oracle

In [ ]:
def write_pseudo_coco(per_image, path):
    imgs = coco_train.loadImgs(coco_train.getImgIds())
    anns, aid = [], 1
    for info in imgs:
        for (x1, y1, x2, y2), s in per_image.get(info['id'], []):
            w, h = float(x2 - x1), float(y2 - y1)
            if w <= 1 or h <= 1:
                continue
            anns.append({'id': aid, 'image_id': info['id'], 'category_id': CAT_ID,
                         'bbox': [float(x1), float(y1), w, h], 'area': w * h, 'iscrowd': 0, 'score': float(s)})
            aid += 1
    json.dump({'images': imgs, 'categories': [{'id': CAT_ID, 'name': 'Car'}], 'annotations': anns}, open(path, 'w'))
    return len(anns)

n1 = write_pseudo_coco(tood_boxes, TOOD_PSEUDO_ANN)
n2 = write_pseudo_coco(fused_boxes, FUSED_PSEUDO_ANN)
true_train, n_imgs = len(coco_train.getAnnIds()), len(coco_train.getImgIds())
print(f"{'set':16} {'boxes':>8} {'per image':>10}   vs oracle true density {true_train/n_imgs:.1f}/image")
print(f"{'source-only':16} {n1:>8} {n1/n_imgs:>10.1f}")
print(f"{'fused':16} {n2:>8} {n2/n_imgs:>10.1f}")
save_result('pseudo_label_stats',
            {'threshold': PSEUDO_THRESHOLD, 'fuse_iou': FUSE_IOU, 'tood_boxes': n1, 'fused_boxes': n2,
             'foundation_added': n2 - n1, 'true_train_boxes': true_train, 'n_images': n_imgs})

set                 boxes  per image   vs oracle true density 25.7/image
source-only         46668       19.1
fused               47520       19.4
saved -> pseudo_label_stats: {'threshold': 0.15, 'fuse_iou': 0.5, 'tood_boxes': 46668, 'fused_boxes': 47520, 'foundation_added': 852, 'true_train_boxes': 63051, 'n_images': 2449, '_saved': '2026-06-17T10:59:15'}


### 3.5 — Write the source anchoring subset
A fixed random subsample of the xView tiles, used to keep the student close to the labeled source distribution during adaptation.

In [ ]:
import random
def write_subset(full_ann, out_ann, fraction, seed=42):
    d = json.load(open(full_ann))
    random.seed(seed)
    k = max(1, int(len(d['images']) * fraction))
    keep = set(random.sample([i['id'] for i in d['images']], k))
    imgs = [i for i in d['images'] if i['id'] in keep]
    anns = [a for a in d['annotations'] if a['image_id'] in keep]
    json.dump({'images': imgs, 'annotations': anns, 'categories': d['categories']}, open(out_ann, 'w'))
    return len(imgs), len(anns)

ti, ta = write_subset(XV_TILES_TRAIN_ANN, XV_TRAIN_SUBSET, SOURCE_FRACTION)
vi, va = write_subset(XV_TILES_VAL_ANN, XV_VAL_SUBSET, SOURCE_FRACTION)
print(f'Source subset (fraction {SOURCE_FRACTION}): train {ti} tiles / {ta} boxes | val {vi} tiles / {va} boxes')
print(f'Approx. source tiles {ti + vi} vs oversampled target {n_imgs * REPEAT_VME} (repeat {REPEAT_VME})')

Source subset (fraction 0.2): train 2130 tiles / 45355 boxes | val 503 tiles / 14873 boxes
Approx. source tiles 2633 vs oversampled target 7347 (repeat 3)


# Section 4 — Foundation-guided student
Student initialised from the Phase 1 checkpoint, trained on a mix of labeled xView tiles (weak augmentation, source anchor) and oversampled fused pseudo-labeled VME (strong augmentation). Validation on VME val is monitoring only — the reported number comes from Section 6.

### 4.1 — Foundation-guided configuration

In [ ]:
fg_config = f"""# TOOD foundation-guided self-training: xView (anchor) + VME (fused pseudo-labels) -> VME
_base_ = 'mmdet::tood/tood_x101-64x4d_fpn_ms-2x_coco.py'

metainfo = dict(classes=('Car',), palette=[(220, 20, 60)])

# source view (labeled xView): weak augmentation
source_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations', with_bbox=True),
    dict(type='RandomFlip', prob=0.5),
    dict(type='PackDetInputs'),
]
# target view (pseudo-labeled VME): strong (student) augmentation
target_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations', with_bbox=True),
    dict(type='PhotoMetricDistortion'),
    dict(type='RandomFlip', prob=0.5),
    dict(type='PackDetInputs'),
]
test_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='Resize', scale=(512, 512), keep_ratio=True),
    dict(type='PackDetInputs'),
]

train_dataloader = dict(
    batch_size=16, num_workers=4,
    dataset=dict(
        _delete_=True, type='ConcatDataset',
        datasets=[
            dict(type='CocoDataset', metainfo=metainfo,
                 ann_file='{XV_TRAIN_SUBSET}',
                 data_prefix=dict(img='{XV_TILES_TRAIN_DIR}'),
                 pipeline=source_pipeline),
            dict(type='CocoDataset', metainfo=metainfo,
                 ann_file='{XV_VAL_SUBSET}',
                 data_prefix=dict(img='{XV_TILES_VAL_DIR}'),
                 pipeline=source_pipeline),
            dict(type='RepeatDataset', times={REPEAT_VME}, dataset=dict(
                 type='CocoDataset', metainfo=metainfo,
                 ann_file='{FUSED_PSEUDO_ANN}',
                 data_prefix=dict(img='{VME_IMGS}/'),
                 pipeline=target_pipeline,
                 filter_cfg=dict(filter_empty_gt=True, min_size=2))),
        ]))

val_dataloader = dict(
    batch_size=2, num_workers=4,
    dataset=dict(_delete_=True, type='CocoDataset', metainfo=metainfo,
                 ann_file='{VME_VAL_ANN}',
                 data_prefix=dict(img='{VME_IMGS}/'),
                 pipeline=test_pipeline, test_mode=True))
val_evaluator = dict(type='CocoMetric', ann_file='{VME_VAL_ANN}', metric=['bbox'])

test_dataloader = dict(
    batch_size=2, num_workers=4,
    dataset=dict(_delete_=True, type='CocoDataset', metainfo=metainfo,
                 ann_file='{VME_TEST_ANN}',
                 data_prefix=dict(img='{VME_IMGS}/'),
                 pipeline=test_pipeline, test_mode=True))
test_evaluator = dict(type='CocoMetric', ann_file='{VME_TEST_ANN}', metric=['bbox'])

model = dict(
    backbone=dict(
        dcn=dict(type='DCNv2', deform_groups=1, fallback_on_stride=False),
        stage_with_dcn=(False, True, True, True),
        init_cfg=dict(type='Pretrained', checkpoint='{BACKBONE}')),
    bbox_head=dict(num_classes=1))

# initialise the student from the Phase 1 source-only checkpoint (overrides the backbone init)
load_from = '{P1_BASELINE_CKPT}'

optim_wrapper = dict(optimizer=dict(type='SGD', lr={SELFTRAIN_LR}, momentum=0.9, weight_decay=1e-4))
train_cfg = dict(max_epochs={SELFTRAIN_EPOCHS}, val_interval=2)
param_scheduler = [
    dict(type='LinearLR', start_factor=0.01, by_epoch=False, begin=0, end=250),
    dict(type='MultiStepLR', begin=0, end={SELFTRAIN_EPOCHS}, by_epoch=True, milestones=[5, 7], gamma=0.1),
]

default_hooks = dict(
    checkpoint=dict(type='CheckpointHook', interval=2, max_keep_ckpts=4, save_last=True),
    logger=dict(type='LoggerHook', interval=50))

randomness = dict(seed=42)

visualizer = dict(
    type='DetLocalVisualizer',
    vis_backends=[
        dict(type='LocalVisBackend'),
        dict(type='WandbVisBackend',
             init_kwargs=dict(project='qcri-vme-uda', name='tood-foundation-guided-st')),
    ])
"""
with open(FG_CFG_PATH, 'w') as f:
    f.write(fg_config)
print('Written:', FG_CFG_PATH)

Written: /content/drive/MyDrive/QCRI-CV/phase3/configs/tood_selftrain_fused.py


### 4.2 — Pre-flight

In [ ]:
import torch
for p in [FUSED_PSEUDO_ANN, TOOD_PSEUDO_ANN, XV_TRAIN_SUBSET, XV_VAL_SUBSET, P1_BASELINE_CKPT]:
    assert os.path.exists(p), f'Missing {p} — run Section 3'
assert 'A100' in torch.cuda.get_device_name(0), 'An A100 runtime is required'
print('Pre-flight passed —', torch.cuda.get_device_name(0))

Pre-flight passed — NVIDIA A100-SXM4-40GB


### 4.3 — Train

In [ ]:
!OPENCV_LOG_LEVEL=SILENT python /content/mmdetection/tools/train.py {FG_CFG_PATH} --work-dir {CKPTS}/foundation_guided --resume

/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
06/17 11:02:12 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 42
    GPU 0: NVIDIA A100-SXM4-40GB
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.4.0+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architectu

---
# Section 5 — Self-training baseline student (ablation)
Identical recipe, initialisation, and source anchoring to Section 4, with the source-only pseudo-labels in place of the fused set. The difference in VME test accuracy between this student and the Section 4 student isolates the effect of foundation guidance.

### 5.1 — Configuration

In [ ]:
st_config = open(FG_CFG_PATH).read() \
    .replace("ann_file='" + FUSED_PSEUDO_ANN + "'", "ann_file='" + TOOD_PSEUDO_ANN + "'") \
    .replace("name='tood-foundation-guided-st'", "name='tood-selftrain-no-foundation'")
with open(ST_CFG_PATH, 'w') as f:
    f.write(st_config)
assert TOOD_PSEUDO_ANN in st_config and FUSED_PSEUDO_ANN not in st_config
print('Written:', ST_CFG_PATH)

Written: /content/drive/MyDrive/QCRI-CV/phase3/configs/tood_selftrain_toodonly.py


### 5.2 — Train

In [ ]:
!OPENCV_LOG_LEVEL=SILENT python /content/mmdetection/tools/train.py {ST_CFG_PATH} --work-dir {CKPTS}/selftrain_baseline --resume

/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
06/17 15:10:30 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 42
    GPU 0: NVIDIA A100-SXM4-40GB
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.4.0+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architectu

---
# Section 6 — Evaluation and comparison
Both students are evaluated on the VME test set at their final epoch with standard inference and the Phase 1 COCO protocol (maxDets = 1000, category id read from ground truth).

### 6.1 — Evaluation helper

In [ ]:
from pycocotools.cocoeval import COCOeval

def evaluate(cfg_path, ckpt_path, tag):
    model = init_detector(cfg_path, ckpt_path, device='cuda:0')
    gt = COCO(VME_TEST_ANN); cat = int(gt.getCatIds()[0])
    preds = []
    for info in tqdm(gt.loadImgs(gt.getImgIds()), desc=tag):
        r = inference_detector(model, os.path.join(VME_IMGS, info['file_name']))
        for (x1, y1, x2, y2), s in zip(r.pred_instances.bboxes.cpu().numpy(),
                                       r.pred_instances.scores.cpu().numpy()):
            preds.append({'image_id': info['id'], 'category_id': cat,
                          'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], 'score': float(s)})
    pf = f'{PREDS}/{tag}.json'; json.dump(preds, open(pf, 'w'))
    dt = gt.loadRes(pf)
    ev = COCOeval(gt, dt, 'bbox'); ev.params.maxDets = [1, 10, 1000]; ev.params.catIds = [cat]
    ev.evaluate(); ev.accumulate(); ev.summarize()
    return float(ev.stats[0]) * 100, float(ev.stats[1]) * 100, len(preds)

### 6.2 — Foundation-guided student

In [ ]:
ckpt = f'{CKPTS}/foundation_guided/epoch_{SELFTRAIN_EPOCHS}.pth'
assert os.path.exists(ckpt), 'Finish Section 4'
mAP, mAP50, n = evaluate(FG_CFG_PATH, ckpt, 'foundation_guided_vme_test')
print(f'\nFoundation-guided VME mAP50: {mAP50:.1f}')
save_result('foundation_guided_vme_test',
            {'mAP50': round(mAP50, 2), 'mAP': round(mAP, 2), 'checkpoint': ckpt, 'n_preds': n})

Loads checkpoint by local backend from path: /content/drive/MyDrive/QCRI-CV/phase3/checkpoints/foundation_guided/epoch_8.pth


/usr/local/lib/python3.12/dist-packages/mmengine/runner/checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filename, map_location=map_l

06/17 18:56:12 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.0.conv2 is upgraded to version 2.
06/17 18:56:12 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.1.conv2 is upgraded to version 2.
06/17 18:56:12 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.2.conv2 is upgraded to version 2.
06/17 18:56:12 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.3.conv2 is upgraded to version 2.
06/17 18:56:12 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.0.conv2 is upgraded to version 2.
06/17 18:56:12 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.1.conv2 is upgraded to version 2.
06/17 18:56:12 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.2.conv2 is upgraded to version 2.
06/17 18:56:12 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.3.conv2 is upgraded to version 2.
06/17 18:56:12 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.4.conv2 is upgraded to version 2.
06/17 18:56:12 - mm

foundation_guided_vme_test: 100%|██████████| 988/988 [15:11<00:00,  1.08it/s]


Loading and preparing results...
DONE (t=0.84s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=40.14s).
Accumulating evaluation results...
DONE (t=0.41s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=1000 ] = 0.399
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=1000 ] = 0.060
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=1000 ] = 0.139
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=1000 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=1000 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.015
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.092
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=1000 ] = 0.247
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | 

### 6.3 — Self-training baseline student

In [ ]:
ckpt = f'{CKPTS}/selftrain_baseline/epoch_{SELFTRAIN_EPOCHS}.pth'
assert os.path.exists(ckpt), 'Finish Section 5'
mAP, mAP50, n = evaluate(ST_CFG_PATH, ckpt, 'selftrain_baseline_vme_test')
print(f'\nSelf-training baseline VME mAP50: {mAP50:.1f}')
save_result('selftrain_baseline_vme_test',
            {'mAP50': round(mAP50, 2), 'mAP': round(mAP, 2), 'checkpoint': ckpt, 'n_preds': n})

Loads checkpoint by local backend from path: /content/drive/MyDrive/QCRI-CV/phase3/checkpoints/selftrain_baseline/epoch_8.pth


/usr/local/lib/python3.12/dist-packages/mmengine/runner/checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filename, map_location=map_l

06/17 19:12:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.0.conv2 is upgraded to version 2.
06/17 19:12:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.1.conv2 is upgraded to version 2.
06/17 19:12:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.2.conv2 is upgraded to version 2.
06/17 19:12:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.3.conv2 is upgraded to version 2.
06/17 19:12:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.0.conv2 is upgraded to version 2.
06/17 19:12:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.1.conv2 is upgraded to version 2.
06/17 19:12:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.2.conv2 is upgraded to version 2.
06/17 19:12:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.3.conv2 is upgraded to version 2.
06/17 19:12:10 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.4.conv2 is upgraded to version 2.
06/17 19:12:10 - mm

selftrain_baseline_vme_test: 100%|██████████| 988/988 [01:50<00:00,  8.92it/s]


Loading and preparing results...
DONE (t=0.79s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=39.66s).
Accumulating evaluation results...
DONE (t=0.38s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=1000 ] = 0.401
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=1000 ] = 0.058
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=1000 ] = 0.138
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=1000 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=1000 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.015
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.092
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=1000 ] = 0.248
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | 

### 6.4 — Comparison

In [ ]:
db = load_results(); p1 = load_phase1(); p2 = load_phase2()
base = p1.get('baseline_vme_test_epoch24', {}).get('mAP50', 'n/a')
orac = p1.get('oracle_vme_test_epoch24', {}).get('mAP50', 'n/a')
gd   = p2.get('gdino_vme_test_best_prompt', {}).get('mAP50', 'n/a')
st   = db.get('selftrain_baseline_vme_test', {}).get('mAP50', 'TBD')
fg   = db.get('foundation_guided_vme_test', {}).get('mAP50', 'TBD')

print('VME test mAP50 — adaptation results\n')
print(f"  {'source-only baseline (xView->VME)':42} {base}")
print(f"  {'Grounding DINO zero-shot':42} {gd}")
print(f"  {'self-training (source pseudo-labels)':42} {st}")
print(f"  {'foundation-guided self-training (ours)':42} {fg}")
print(f"  {'oracle (VME labels, upper bound)':42} {orac}")
if isinstance(st, (int, float)) and isinstance(fg, (int, float)):
    print(f"\n  foundation guidance: {fg - st:+.1f} mAP50 over self-training alone")
    if isinstance(base, (int, float)):
        print(f"  adaptation gain    : {fg - base:+.1f} mAP50 over source-only baseline")

VME test mAP50 — adaptation results

  source-only baseline (xView->VME)          38.23
  Grounding DINO zero-shot                   14.68
  self-training (source pseudo-labels)       40.05
  foundation-guided self-training (ours)     39.92
  oracle (VME labels, upper bound)           55.58

  foundation guidance: -0.1 mAP50 over self-training alone
  adaptation gain    : +1.7 mAP50 over source-only baseline


### Section 6.4 - xView re-evaluation + PD/H for the adapted models

In [ ]:
# 6.5a — local xView copy (throttle-safe, skips files already present)
import shutil, os, time

XVIEW_RAW_IMGS  = f'{BASE}/xview/train_images'
XVIEW_TEST_ANN  = f'{P1}/annotations/xview/xview_car_test.json'
XVIEW_IMGS_LOCAL = '/content/xview_images'

os.makedirs(XVIEW_IMGS_LOCAL, exist_ok=True)
already = set(os.listdir(XVIEW_IMGS_LOCAL))
tifs = sorted([f for f in os.listdir(XVIEW_RAW_IMGS) if f.endswith('.tif')])
for i, f in enumerate(tifs):
    if f in already:
        continue
    shutil.copy2(f'{XVIEW_RAW_IMGS}/{f}', f'{XVIEW_IMGS_LOCAL}/{f}')
    if i % 50 == 0:
        print(f'{i}/{len(tifs)}'); time.sleep(1)
print(f'Done — {len(os.listdir(XVIEW_IMGS_LOCAL))} files')

0/846
50/846
100/846
150/846
200/846
250/846
300/846
350/846
400/846
450/846
500/846
550/846
600/846
650/846
700/846
750/846
800/846
Done — 846 files


In [ ]:
# 6.5b — evaluation helper (note: maxDets includes 100 so mAP@[.50:.95] is no longer broken)
import logging
import os, json
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

logging.getLogger('sahi').setLevel(logging.ERROR)

def evaluate_xview(cfg_path, ckpt_path, tag):
    dm = AutoDetectionModel.from_pretrained(model_type='mmdet', model_path=ckpt_path,
                                            config_path=cfg_path, confidence_threshold=0.05, device='cuda:0')
    gt = COCO(XVIEW_TEST_ANN); cat = int(gt.getCatIds()[0])
    preds = []
    for info in tqdm(gt.loadImgs(gt.getImgIds()), desc=tag):
        res = get_sliced_prediction(os.path.join(XVIEW_IMGS_LOCAL, info['file_name']), dm,
                                    slice_height=512, slice_width=512,
                                    overlap_height_ratio=0.2, overlap_width_ratio=0.2,
                                    perform_standard_pred=True, verbose=0)
        for p in res.object_prediction_list:
            b = p.bbox
            preds.append({'image_id': info['id'], 'category_id': cat,
                          'bbox': [b.minx, b.miny, b.maxx - b.minx, b.maxy - b.miny], 'score': float(p.score.value)})
    pf = f'{PREDS}/{tag}.json'; json.dump(preds, open(pf, 'w'))
    dt = gt.loadRes(pf)
    ev = COCOeval(gt, dt, 'bbox'); ev.params.maxDets = [1, 10, 1000, 100]; ev.params.catIds = [cat]
    ev.evaluate(); ev.accumulate(); ev.summarize()
    return float(ev.stats[0]) * 100, float(ev.stats[1]) * 100, len(preds)

mAP, mAP50, n = evaluate_xview(P1_BASELINE_CFG, f'{CKPTS}/foundation_guided/epoch_8.pth', 'foundation_guided_xview_test')
save_result('foundation_guided_xview_test', {'mAP50': round(mAP50, 2), 'mAP': round(mAP, 2), 'n_preds': n})

mAP, mAP50, n = evaluate_xview(P1_BASELINE_CFG, f'{CKPTS}/selftrain_baseline/epoch_8.pth', 'selftrain_baseline_xview_test')
save_result('selftrain_baseline_xview_test', {'mAP50': round(mAP50, 2), 'mAP': round(mAP, 2), 'n_preds': n})

Loads checkpoint by local backend from path: /content/drive/MyDrive/QCRI-CV/phase3/checkpoints/foundation_guided/epoch_8.pth


/usr/local/lib/python3.12/dist-packages/mmengine/runner/checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filename, map_location=map_l

06/18 10:03:03 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.0.conv2 is upgraded to version 2.
06/18 10:03:03 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.1.conv2 is upgraded to version 2.
06/18 10:03:03 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.2.conv2 is upgraded to version 2.
06/18 10:03:03 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.3.conv2 is upgraded to version 2.
06/18 10:03:03 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.0.conv2 is upgraded to version 2.
06/18 10:03:03 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.1.conv2 is upgraded to version 2.
06/18 10:03:03 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.2.conv2 is upgraded to version 2.
06/18 10:03:03 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.3.conv2 is upgraded to version 2.
06/18 10:03:03 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.4.conv2 is upgraded to version 2.
06/18 10:03:03 - mm

/usr/local/lib/python3.12/dist-packages/mmengine/visualization/visualizer.py:196: UserWarning: Failed to add <class 'mmengine.visualization.vis_backend.LocalVisBackend'>, please provide the `save_dir` argument.
  warnings.warn(f'Failed to add {vis_backend.__class__}, '
/usr/local/lib/python3.12/dist-packages/mmengine/visualization/visualizer.py:196: UserWarning: Failed to add <class 'mmengine.visualization.vis_backend.WandbVisBackend'>, please provide the `save_dir` argument.
  warnings.warn(f'Failed to add {vis_backend.__class__}, '


Done (t=0.10s)
creating index...
index created!


...

06/18 10:30:50 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.0.conv2 is upgraded to version 2.
06/18 10:30:50 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.1.conv2 is upgraded to version 2.
06/18 10:30:50 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.2.conv2 is upgraded to version 2.
06/18 10:30:50 - mmengine - INFO - ModulatedDeformConvPack backbone.layer2.3.conv2 is upgraded to version 2.
06/18 10:30:50 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.0.conv2 is upgraded to version 2.
06/18 10:30:50 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.1.conv2 is upgraded to version 2.
06/18 10:30:50 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.2.conv2 is upgraded to version 2.
06/18 10:30:50 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.3.conv2 is upgraded to version 2.
06/18 10:30:50 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.4.conv2 is upgraded to version 2.
06/18 10:30:50 - mm

/usr/local/lib/python3.12/dist-packages/mmengine/visualization/visualizer.py:196: UserWarning: Failed to add <class 'mmengine.visualization.vis_backend.LocalVisBackend'>, please provide the `save_dir` argument.
  warnings.warn(f'Failed to add {vis_backend.__class__}, '
/usr/local/lib/python3.12/dist-packages/mmengine/visualization/visualizer.py:196: UserWarning: Failed to add <class 'mmengine.visualization.vis_backend.WandbVisBackend'>, please provide the `save_dir` argument.
  warnings.warn(f'Failed to add {vis_backend.__class__}, '


Done (t=0.10s)
creating index...
index created!


selftrain_baseline_xview_test: 100%|██████████| 167/167 [18:49<00:00,  6.77s/it]


Loading and preparing results...
DONE (t=2.65s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=516.56s).
Accumulating evaluation results...
DONE (t=0.60s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.073
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.185
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.037
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.073
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.002
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.016
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.105
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDet

In [ ]:
# 6.5c — PD/H for both adapted models, compared to the source-only baseline
db = load_results()
def pd_h(idd, ood): return 100*(idd-ood)/idd, 2*ood*idd/(ood+idd)

for tag, ood_key in [('foundation-guided', 'foundation_guided'), ('self-training-only', 'selftrain_baseline')]:
    idd = db[f'{ood_key}_xview_test']['mAP50']
    ood = db[f'{ood_key}_vme_test']['mAP50']
    pd50, h50 = pd_h(idd, ood)
    print(f'{tag:20} ID {idd:.1f} | OOD {ood:.1f} | PD {pd50:.1f}% | H {h50:.1f}')
    save_result(f'{ood_key}_pd_h', {'PD_mAP50': round(pd50, 1), 'H_mAP50': round(h50, 1)})

p1 = load_phase1()['baseline_pd_h']
print(f"\nsource-only baseline   PD {p1['PD_mAP50']}% | H {p1['H_mAP50']}")

foundation-guided    ID 18.7 | OOD 39.9 | PD -113.7% | H 25.5
saved -> foundation_guided_pd_h: {'PD_mAP50': -113.7, 'H_mAP50': 25.5, '_saved': '2026-06-18T10:59:48'}
self-training-only   ID 18.5 | OOD 40.0 | PD -116.5% | H 25.3
saved -> selftrain_baseline_pd_h: {'PD_mAP50': -116.5, 'H_mAP50': 25.3, '_saved': '2026-06-18T10:59:48'}

source-only baseline   PD 34.9% | H 46.3


In [ ]:
# Recomputes mAP@[0.50:0.95] using the correct maxDets list
import json, os
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

BASE = '/content/drive/MyDrive/QCRI-CV'
P1, P2, P3 = f'{BASE}/phase1', f'{BASE}/phase2', f'{BASE}/phase3'
VME_TEST = f'{P1}/annotations/vme/test.json'
XV_TEST  = f'{P1}/annotations/xview/xview_car_test.json'

FIXES = [
    (f'{P1}/results/phase1_results.json', 'baseline_vme_test_epoch24', VME_TEST, f'{P1}/results/preds/vme_test_epoch24.json'),
    (f'{P1}/results/phase1_results.json', 'baseline_xview_test_epoch24', XV_TEST, f'{P1}/results/preds/xview_test_epoch24.json'),
    (f'{P1}/results/phase1_results.json', 'oracle_vme_test_epoch24', VME_TEST, f'{P1}/results/preds/oracle_vme_test_epoch24.json'),
    (f'{P1}/results/phase1_results.json', 'oracle_sft_vme_test_epoch24', VME_TEST, f'{P1}/results/preds/oracle_sft_vme_test_epoch24.json'),
    (f'{P2}/results/phase2_results.json', 'gdino_vme_test_best_prompt', VME_TEST, f'{P2}/preds/vme_gdino_car.json'),
    (f'{P2}/results/phase2_results.json', 'gdino_xview_test', XV_TEST, f'{P2}/preds/xview_gdino.json'),
    (f'{P3}/results/phase3_results.json', 'foundation_guided_vme_test', VME_TEST, f'{P3}/results/preds/foundation_guided_vme_test.json'),
    (f'{P3}/results/phase3_results.json', 'selftrain_baseline_vme_test', VME_TEST, f'{P3}/results/preds/selftrain_baseline_vme_test.json'),
]

for results_path, key, gt_path, pred_path in FIXES:
    if not (os.path.exists(results_path) and os.path.exists(pred_path)):
        print(f'[SKIP] {key}: file missing'); continue
    gt = COCO(gt_path); cat = int(gt.getCatIds()[0])
    dt = gt.loadRes(pred_path)
    ev = COCOeval(gt, dt, 'bbox'); ev.params.maxDets = [1, 10, 1000, 100]; ev.params.catIds = [cat]
    ev.evaluate(); ev.accumulate(); ev.summarize()
    new_map = round(float(ev.stats[0]) * 100, 2)
    db = json.load(open(results_path))
    old_map = db[key].get('mAP')
    db[key]['mAP'] = new_map
    json.dump(db, open(results_path, 'w'), indent=2)
    print(f'[FIXED] {key}: mAP {old_map} -> {new_map}')

loading annotations into memory...
Done (t=0.74s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.38s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=32.61s).
Accumulating evaluation results...
DONE (t=0.46s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.134
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.382
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.056
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.134
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.015
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.089
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxD

---
# Section 7 — Recorded results

In [ ]:
print(json.dumps(load_results(), indent=2))

{
  "pseudo_threshold_selection": {
    "threshold": 0.15,
    "val_precision": 0.546,
    "val_recall": 0.434,
    "val_f1": 0.483,
    "_saved": "2026-06-17T10:32:26"
  },
  "pseudo_label_stats": {
    "threshold": 0.15,
    "fuse_iou": 0.5,
    "tood_boxes": 46668,
    "fused_boxes": 47520,
    "foundation_added": 852,
    "true_train_boxes": 63051,
    "n_images": 2449,
    "_saved": "2026-06-17T10:59:15"
  },
  "foundation_guided_vme_test": {
    "mAP50": 39.92,
    "mAP": -100.0,
    "checkpoint": "/content/drive/MyDrive/QCRI-CV/phase3/checkpoints/foundation_guided/epoch_8.pth",
    "n_preds": 82539,
    "_saved": "2026-06-17T19:12:07"
  },
  "selftrain_baseline_vme_test": {
    "mAP50": 40.05,
    "mAP": -100.0,
    "checkpoint": "/content/drive/MyDrive/QCRI-CV/phase3/checkpoints/selftrain_baseline/epoch_8.pth",
    "n_preds": 76235,
    "_saved": "2026-06-17T19:14:43"
  }
}


In [ ]:
from google.colab import runtime
runtime.unassign()